In [1]:
from dataclasses import dataclass, field

from dowhy import CausalModel
import pandas as pd

c:\Users\Roma\.virtualenvs\causal-promo-opt-N4J8iauW\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
@dataclass
class ModelResult:
    label: str
    data: any
    treatment: any
    outcome: str
    confounders: list
    method_params: dict

    model: any = field(init=False)
    estimand: any = field(init=False)
    estimate: any = field(init=False)

    def fit(self):
        self.model = CausalModel(
            data = self.data,
            treatment = self.treatment,
            outcome = self.outcome,
            common_causes = self.confounders
        )

        self.estimand = self.model.identify_effect()

        self.estimate = self.model.estimate_effect(
            self.estimand,
            **self.method_params
        )

    def run_refuters(self):
        results = []

        refuters = [
            ("data_subset", {
                "method_name": "data_subset_refuter",
                "subset_fraction": 0.8,
                "num_simulations": 100
            }),
            ("random_common_cause", {
                "method_name": "random_common_cause",
                "num_simulations": 100
            }),
            ("random_treatment", {
                "method_name": "placebo_treatment_refuter",
                "num_simulations": 100
            }),
            ("random_outcome", {
                "method_name": "dummy_outcome_refuter",
                "num_simulations": 100
            })
        ]

        for name, params in refuters:
            refute = self.model.refute_estimate(
                self.estimand,
                self.estimate,
                **params
            )

            results.append({
                "treatment": self.label,
                "refuter": name,
                "original_effect": refute.estimated_effect,
                "new_effect": refute.new_effect,
                "delta": refute.new_effect - refute.estimated_effect,
                "p_value": refute.refutation_result.get("p_value", None)
            })

        return results